In [1]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

c:\Users\Masum\anaconda3\envs\langagent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain.agents import create_react_agent, AgentExecutor

In [3]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [4]:
search_tool = TavilySearchResults(max_results=2)

In [5]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [6]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'url': 'https://www.crescendo.ai/news/latest-ai-news-and-updates',
  'content': 'Date: May 21, 2026\n\nSummary: SoundHound AI announced the acquisition of conversational AI platform LivePerson, combining its voice and audio recognition capabilities with LivePerson\'s digital messaging products. Management has set 2027 revenue targets of $350 million to $400 million and identified a $500 million cross-selling opportunity across the combined customer base. The deal positions SoundHound to compete more directly with Microsoft, Google, and Amazon across enterprise contact centers, healthcare, and financial services.\n\nSource:Simply Wall St ↗\n\n### (AI News) Nvidia\'s Memory Costs Soar 485% as Latest AI Systems Now Cost $7.8 Million to Build\n\nDate: May 21, 2026 [...] Source:425 Business\n\n### (AI Innovation) Microsoft Launches $2.5 Billion Frontier Company for Enterprise AI Deployment\n\nDate: July 2, 2026\n\nSummary: Microsoft announced Microsoft Frontier Company, backed by $2.5 bil

In [7]:
# ==========================================
# LLM
# ==========================================

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    api_key=GOOGLE_API_KEY
)

In [ ]:
response = llm.invoke("Tell me about masum")
response

AIMessage(content='Because "Masum Billah" (also spelled Mohammed Masum Billah, Md. Masum Billah, etc.) is a very common Arabic and Bengali name, there are several notable individuals who go by this name. \n\nTo give you the most accurate information, here are the most prominent figures known as **Masum Billah**:\n\n### 1. Masum Billah (Bangladesh Educationist & Writer)\nOne of the most well-known figures with this name in Bangladesh is **Masum Billah**, a prominent educationist, writer, and activist. \n* **Role:** He is widely recognized for his contributions to English teaching and education reform in Bangladesh. \n* **Work:** He has worked extensively with organizations like BRAC Education Programme and various teacher-training institutions. He is a regular contributor to national Bangladeshi newspapers, writing opinion pieces and articles focusing on the flaws and potential improvements in the country\'s education system, curriculum, and teacher training.\n\n### 2. Dr. Mohammad Masu

In [9]:
# ==========================================
# PROMPT
# ==========================================

prompt = hub.pull("hwchase17/react")

c:\Users\Masum\anaconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [10]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [22]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

In [23]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [24]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [33]:
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({
    "input": (
        "Tell me about dengue fever news updates in Bangladesh 2026 "
        "and give me the current weather in Dhaka, Bangladesh"
        "tell me about 1971 in bangladesh"
    )
})




> Entering new AgentExecutor chain...
Thought:
Action: tavily_search_results_json
Action Input: dengue fever Bangladesh 2026 news
[{'url': 'https://reliefweb.int/disaster/ep-2026-000174-bgd', 'content': 'Bangladesh is experiencing an active and worsening dengue outbreak. By 14 September 2026, 53,570 dengue patients had been hospitalized and 157 deaths recorded, compared with 37,891 admissions and 150 deaths in 2025 [...] The highest burden is currently concentrated in Dhaka, Khulna and Chattogram, while increasing trends have been identified in Pirojpur, Cox’s Bazar and Barguna, with Netrokona also showing an upward trend. The continued increase in cases is placing additional pressure on health facilities, including bed capacity, clinical staff, blood and platelet supplies, and other essential resources. (IFRC, 23 Sep 2026) [...] In Cox’s Bazar, there was a two-fold drop in monthly RDT-positive dengue cases in September: 604 cases compared to 1,218 cases reported in the previous mont

In [34]:
print(response["output"])

**Dengue Fever News Updates in Bangladesh (2026):**
Bangladesh has been experiencing an active and worsening dengue outbreak. By mid-September 2026, reports indicated that over 53,000 dengue patients had been hospitalized and more than 150 deaths recorded, showing a significant increase compared to previous years. The highest burden is concentrated in divisions like Dhaka, Khulna, and Chattogram, with severe pressure on health facilities, bed capacities, and blood/platelet supplies. Organizations like the Bangladesh Red Crescent Society (BDRCS), IFRC, and WHO have been scaling up emergency responses to support affected communities and Rohingya camps.

**Current Weather in Dhaka, Bangladesh:**
* **Temperature:** 33°C
* **Weather:** Sunny
* **Humidity:** 37%

**Overview of 1971 in Bangladesh:**
The year 1971 marks a pivotal moment in Bangladeshi history—the **Bangladesh Liberation War**. Following years of political, cultural, and economic discrimination by West Pakistan against East Pak